# Hack-A-Roo 2026, AI for Quantum track
# Notebook 1: QAOA warm start

**Problem.** QAOA solves Max-Cut by optimizing a handful of circuit angles. Every new graph starts the optimizer from scratch. Can a classical model predict good starting angles from cheap graph features, so the optimizer does less work?

**What this notebook already does.** It generates random graphs, runs QAOA on each one with a plain statevector simulation, records the best angles the optimizer found, trains a regressor from graph features to those angles, and compares warm starts against random starts on graphs whose topology the regressor never saw.

**Metrics.** Three numbers, all on held-out graphs. First, objective evaluations until the optimizer terminates (COBYLA counts evaluations, not iterations, and it stops at a tolerance or at a budget cap). Second, objective evaluations until the run first reaches a fixed cut-quality target, which is the fair speed comparison because a random start and a warm start otherwise terminate at different qualities. Third, the approximation ratio at the starting angles before any optimization.

**Hypothesis to test.** A warm start reaches the quality target in fewer evaluations than a random start. The notebook does not assume this; the poster reports what happened, including a tie or a loss.

**Deliverable for the poster.** One figure with the three metrics and the numbers behind it. Then whichever shred-out you pick at the bottom.

Run every cell top to bottom once. The base run takes one to two minutes. Then go to the Shred-outs section.

In [ ]:
# CONFIG CELL. Every constant, seed, and path lives here. Nothing is hardcoded downstream.

RANDOM_SEED = 42

# Graph generation
NUMBER_OF_GRAPHS = 300
NODE_COUNT = 8
EDGE_PROBABILITY_LOW = 0.3     # each graph draws its own edge probability from this range
EDGE_PROBABILITY_HIGH = 0.8

# QAOA
QAOA_LAYERS = 1                # p. Angles per graph = 2 * QAOA_LAYERS
OPTIMIZER_MAX_EVALUATIONS = 300    # COBYLA's maxiter is a cap on objective evaluations
OPTIMIZER_TOLERANCE = 1e-3
RANDOM_RESTARTS_PER_GRAPH = 3  # best of these restarts is the label. It is the best angle set FOUND, not a proven optimum.

# Angle symmetries for unweighted Max-Cut. See the circuit cell for why these periods hold.
GAMMA_PERIOD = 2 * 3.141592653589793    # 2 pi
BETA_PERIOD = 3.141592653589793 / 2     # pi/2

# Learning
TEST_FRACTION = 0.25           # fraction of ISOMORPHISM CLASSES held out, so no test graph has a training twin
RANDOM_FOREST_TREES = 200

# Evaluation
EVALUATION_RANDOM_STARTS = 3   # independent single random starts per test graph, each compared against the one warm start
TARGET_FRACTION_OF_BEST = 0.98 # a run "reaches the target" when its expected cut first exceeds this fraction of the graph's label cut

# Output
OUTPUT_PATH = "qaoa_warm_start_dataset.pkl"
FIGURE_PATH = "qaoa_warm_start_figure.png"

In [ ]:
# Install. Skip this cell if your environment already has these. Tested with qiskit 2.6, scikit-learn 1.8 and 1.9, networkx 3.7.
# qiskit-aer is only needed for shred-out 7.
!pip install -q "qiskit[visualization]" "qiskit-aer>=0.17" "scikit-learn>=1.5" "networkx>=3.2" matplotlib

In [ ]:
import time
import pickle
import warnings
import numpy as np
import scipy
import sklearn
import qiskit
import networkx as nx
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit

random_generator = np.random.default_rng(RANDOM_SEED)
package_versions = {"numpy": np.__version__, "scipy": scipy.__version__, "qiskit": qiskit.__version__,
                    "scikit-learn": sklearn.__version__, "networkx": nx.__version__}
print("imports done", package_versions)

In [ ]:
# Generate random graphs and compute cheap classical features for each one.
# The features are what the regressor sees. The best angles found are what it predicts.

FEATURE_NAMES = [
    "edge_count",
    "density",
    "mean_degree",
    "degree_std",
    "triangle_count",
    "average_clustering",
    "algebraic_connectivity",
]

graphs = []
graph_edge_lists = []
graph_features = []

for graph_index in range(NUMBER_OF_GRAPHS):
    edge_probability = random_generator.uniform(EDGE_PROBABILITY_LOW, EDGE_PROBABILITY_HIGH)
    graph_seed = int(random_generator.integers(0, 2**31))
    graph = nx.gnp_random_graph(NODE_COUNT, edge_probability, seed=graph_seed)
    # Isolated nodes make the Max-Cut trivially separable and distort the features. Reject those graphs.
    while min(dict(graph.degree()).values()) == 0:
        graph_seed = int(random_generator.integers(0, 2**31))
        graph = nx.gnp_random_graph(NODE_COUNT, edge_probability, seed=graph_seed)

    degree_values = np.array([degree for node, degree in graph.degree()])
    laplacian_eigenvalues = np.sort(np.linalg.eigvalsh(nx.laplacian_matrix(graph).toarray()))
    feature_row = [
        graph.number_of_edges(),
        nx.density(graph),
        degree_values.mean(),
        degree_values.std(),
        sum(nx.triangles(graph).values()) / 3,
        nx.average_clustering(graph),
        laplacian_eigenvalues[1],
    ]
    graphs.append(graph)
    graph_edge_lists.append(list(graph.edges()))
    graph_features.append(feature_row)

graph_features = np.array(graph_features)
assert graph_features.shape == (NUMBER_OF_GRAPHS, len(FEATURE_NAMES))
print("generated", NUMBER_OF_GRAPHS, "graphs on", NODE_COUNT, "nodes")
print("edge counts range from", int(graph_features[:, 0].min()), "to", int(graph_features[:, 0].max()))

In [ ]:
# Group the graphs by isomorphism class, so the train/test split can hold out TOPOLOGIES rather than samples.
# Random 8-node graphs repeat: two draws from the generator are often the same graph with relabeled nodes.
# A Weisfeiler-Lehman hash sorts graphs into buckets quickly; an exact isomorphism test inside each bucket settles it.

def isomorphism_class_ids(graph_list):
    """Assigns an integer class id to every graph so that two graphs share an id exactly when they are isomorphic"""
    hash_buckets = {}
    class_representatives = []
    class_ids = np.full(len(graph_list), -1)
    for graph_index, graph in enumerate(graph_list):
        graph_hash = nx.weisfeiler_lehman_graph_hash(graph)
        for candidate_class_id in hash_buckets.get(graph_hash, []):
            if nx.is_isomorphic(graph, class_representatives[candidate_class_id]):
                class_ids[graph_index] = candidate_class_id
                break
        if class_ids[graph_index] == -1:
            new_class_id = len(class_representatives)
            class_representatives.append(graph)
            hash_buckets.setdefault(graph_hash, []).append(new_class_id)
            class_ids[graph_index] = new_class_id
    assert (class_ids >= 0).all()
    return class_ids


# networkx prints a one-time notice that its hash values changed in version 3.5. The hash is only a bucket key
# inside this one run, so the change is irrelevant here and the notice is silenced.
warnings.filterwarnings("ignore", message="The hashes produced for graphs without node or edge attributes changed")
start_time = time.time()
graph_class_ids = isomorphism_class_ids(graphs)
print(f"{len(np.unique(graph_class_ids))} isomorphism classes among {NUMBER_OF_GRAPHS} graphs, found in {time.time() - start_time:.1f}s")

In [ ]:
# The QAOA circuit, built gate by gate so every element is visible. Every helper takes node_count as an argument so
# shred-out 2 (transfer to larger graphs) can reuse them unchanged.
#
# Max-Cut cost for a bitstring z: number of edges (i, j) with z_i != z_j.
# As an operator: C = sum over edges of (1 - Z_i Z_j) / 2.
# QAOA layer k applies exp(-i gamma_k C) then exp(-i beta_k sum_j X_j).
#   exp(-i gamma (1 - Z_i Z_j) / 2) is a global phase times exp(+i gamma Z_i Z_j / 2),
#   and qiskit's rzz(theta) is exp(-i theta Z Z / 2), so one edge is rzz(-gamma).
#   exp(-i beta X) is qiskit's rx(2 beta).
#
# Symmetries of the expected cut used to wrap the labels into one fundamental region (all three hold for every
# unweighted graph and every p, and the sanity-check block below checks them numerically on the first graph):
#   gamma_k -> gamma_k + 2 pi     every edge term exp(i gamma Z Z / 2) picks up exp(i pi Z Z) = -1, a global phase.
#   beta_k  -> beta_k + pi/2      applies X to every qubit at layer k. X X commutes with every later Z Z and with
#                                 every later rx, so the X's slide to the end and flip every bit. Max-Cut is bit-flip symmetric.
#   (all gammas, all betas) -> (-all gammas, -all betas)    complex conjugation of the whole circuit.
# gamma_k -> gamma_k + pi is NOT a symmetry in general: it applies Z to every odd-degree vertex, which flips the sign
# of every LATER beta on those vertices. Graphs whose degrees all share a parity get an extra symmetry from it.
# That is shred-out 4.

def build_qaoa_circuit(edge_list, gammas, betas, node_count):
    """Builds the p-layer QAOA circuit for Max-Cut on the given edges"""
    assert len(gammas) == len(betas) == QAOA_LAYERS
    circuit = QuantumCircuit(node_count)
    circuit.h(range(node_count))
    for layer_index in range(QAOA_LAYERS):
        for node_i, node_j in edge_list:
            circuit.rzz(-gammas[layer_index], node_i, node_j)
        circuit.rx(2 * betas[layer_index], range(node_count))
    return circuit


def cut_value_of_every_bitstring(edge_list, node_count):
    """Returns a length 2^n array, entry k is the cut value of the bitstring with statevector index k"""
    # qiskit orders the statevector so that qubit 0 is the least significant bit of the index
    bitstring_indices = np.arange(2 ** node_count)
    cut_values = np.zeros(2 ** node_count)
    for node_i, node_j in edge_list:
        bit_i = (bitstring_indices >> node_i) & 1
        bit_j = (bitstring_indices >> node_j) & 1
        cut_values += bit_i != bit_j
    return cut_values


def expected_cut(angles, edge_list, cut_values, node_count):
    """Exact expected cut value of the QAOA state at these angles, angles = [gammas..., betas...]"""
    gammas = angles[:QAOA_LAYERS]
    betas = angles[QAOA_LAYERS:]
    circuit = build_qaoa_circuit(edge_list, gammas, betas, node_count)
    probabilities = Statevector(circuit).probabilities()
    assert probabilities.shape == cut_values.shape
    return float(probabilities @ cut_values)


def wrap_angles(angles):
    """Wraps gammas into [0, 2 pi), betas into [0, pi/2), then uses the whole-sequence sign flip to put the first gamma in [0, pi]"""
    gammas = np.mod(angles[:QAOA_LAYERS], GAMMA_PERIOD)
    betas = np.mod(angles[QAOA_LAYERS:], BETA_PERIOD)
    if gammas[0] > np.pi:
        gammas = np.mod(-gammas, GAMMA_PERIOD)
        betas = np.mod(-betas, BETA_PERIOD)
    return np.concatenate([gammas, betas])


def symmetry_aware_angle_error(predicted_angles, true_angles):
    """Smallest total circular distance between two angle sets, allowing the periods above and the whole-sequence sign flip"""
    periods = np.array([GAMMA_PERIOD] * QAOA_LAYERS + [BETA_PERIOD] * QAOA_LAYERS)
    errors = []
    for sign in (1, -1):
        difference = np.mod(predicted_angles - sign * true_angles, periods)
        circular_distance = np.minimum(difference, periods - difference)
        errors.append(circular_distance)
    errors = np.array(errors)                                   # (2, angles)
    best_sign = np.argmin(errors.sum(axis=1))
    return errors[best_sign]


def optimize_angles(starting_angles, edge_list, cut_values, node_count, target_cut=None):
    """Runs COBYLA from the starting angles and returns a dict: wrapped best angles, best cut seen, evaluations until termination, evaluations until the cut first exceeds target_cut (None if never), and the optimizer's status message"""
    evaluation_log = []

    def negative_expected_cut(angles):
        value = expected_cut(angles, edge_list, cut_values, node_count)
        evaluation_log.append(value)
        return -value

    result = minimize(negative_expected_cut, starting_angles, method="COBYLA",
                      options={"maxiter": OPTIMIZER_MAX_EVALUATIONS, "tol": OPTIMIZER_TOLERANCE})
    running_best = np.maximum.accumulate(evaluation_log)
    evaluations_to_target = None
    if target_cut is not None:
        reached = np.nonzero(running_best >= target_cut)[0]
        if len(reached) > 0:
            evaluations_to_target = int(reached[0]) + 1
    return {
        "angles": wrap_angles(result.x),
        "best_cut": float(running_best[-1]),
        "evaluations": int(result.nfev),
        "evaluations_to_target": evaluations_to_target,
        "hit_evaluation_cap": bool(result.nfev >= OPTIMIZER_MAX_EVALUATIONS),
        "message": str(result.message),
    }


# Sanity checks on one graph: the cut values look right, and the three symmetries above really hold.
example_edge_list = graph_edge_lists[0]
example_cut_values = cut_value_of_every_bitstring(example_edge_list, NODE_COUNT)
assert example_cut_values.max() <= len(example_edge_list)
example_angles = random_generator.uniform(0, 1, size=2 * QAOA_LAYERS)
example_value = expected_cut(example_angles, example_edge_list, example_cut_values, NODE_COUNT)
print("graph 0 has", len(example_edge_list), "edges, brute-force max cut", int(example_cut_values.max()))
print("expected cut at a random angle:", round(example_value, 3))

shifted_gamma = example_angles.copy()
shifted_gamma[:QAOA_LAYERS] += GAMMA_PERIOD
shifted_beta = example_angles.copy()
shifted_beta[QAOA_LAYERS:] += BETA_PERIOD
assert abs(expected_cut(shifted_gamma, example_edge_list, example_cut_values, NODE_COUNT) - example_value) < 1e-9
assert abs(expected_cut(shifted_beta, example_edge_list, example_cut_values, NODE_COUNT) - example_value) < 1e-9
assert abs(expected_cut(-example_angles, example_edge_list, example_cut_values, NODE_COUNT) - example_value) < 1e-9
assert abs(expected_cut(wrap_angles(example_angles + 5.0), example_edge_list, example_cut_values, NODE_COUNT)
           - expected_cut(example_angles + 5.0, example_edge_list, example_cut_values, NODE_COUNT)) < 1e-9
assert np.allclose(symmetry_aware_angle_error(wrap_angles(example_angles + 5.0), example_angles + 5.0), 0)
print("the three angle symmetries hold on graph 0, wrap_angles preserves the expected cut, and the angle error sees wrapped angles as identical")
print(build_qaoa_circuit(example_edge_list, example_angles[:QAOA_LAYERS], example_angles[QAOA_LAYERS:], NODE_COUNT).draw(fold=120))

In [ ]:
# Label every graph: best angles over several random restarts, plus the brute-force max cut for the ratio.
# This is the slow cell. It prints progress so a stall is distinguishable from slow work.

best_angles_found = np.zeros((NUMBER_OF_GRAPHS, 2 * QAOA_LAYERS))
best_expected_cut_found = np.zeros(NUMBER_OF_GRAPHS)
brute_force_max_cut = np.zeros(NUMBER_OF_GRAPHS)
label_evaluations = np.zeros(NUMBER_OF_GRAPHS)   # total objective evaluations spent labeling each graph
label_runs_at_cap = 0

start_time = time.time()
for graph_index in range(NUMBER_OF_GRAPHS):
    edge_list = graph_edge_lists[graph_index]
    cut_values = cut_value_of_every_bitstring(edge_list, NODE_COUNT)
    brute_force_max_cut[graph_index] = cut_values.max()

    for restart_index in range(RANDOM_RESTARTS_PER_GRAPH):
        starting_angles = np.concatenate([
            random_generator.uniform(0, GAMMA_PERIOD, size=QAOA_LAYERS),
            random_generator.uniform(0, BETA_PERIOD, size=QAOA_LAYERS),
        ])
        run = optimize_angles(starting_angles, edge_list, cut_values, NODE_COUNT)
        label_evaluations[graph_index] += run["evaluations"]
        label_runs_at_cap += run["hit_evaluation_cap"]
        if run["best_cut"] > best_expected_cut_found[graph_index]:
            best_expected_cut_found[graph_index] = run["best_cut"]
            best_angles_found[graph_index] = run["angles"]

    if (graph_index + 1) % 25 == 0:
        elapsed_seconds = time.time() - start_time
        seconds_per_graph = elapsed_seconds / (graph_index + 1)
        remaining_seconds = seconds_per_graph * (NUMBER_OF_GRAPHS - graph_index - 1)
        print(f"labeled {graph_index + 1}/{NUMBER_OF_GRAPHS}  elapsed {elapsed_seconds:.0f}s  remaining about {remaining_seconds:.0f}s")

label_approximation_ratio = best_expected_cut_found / brute_force_max_cut
print("mean approximation ratio of the labels:", round(label_approximation_ratio.mean(), 4))
print("mean objective evaluations spent per graph:", round(label_evaluations.mean(), 1))
print(f"labeling runs that stopped at the evaluation cap: {label_runs_at_cap} of {NUMBER_OF_GRAPHS * RANDOM_RESTARTS_PER_GRAPH}")

In [ ]:
# Split by isomorphism class, then fit the baseline: a random forest from graph features to the best angles found.
# This is deliberately the simplest thing that works. Beating it is the point.

group_splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=RANDOM_SEED)
train_indices, test_indices = next(group_splitter.split(graph_features, groups=graph_class_ids))
assert len(np.intersect1d(graph_class_ids[train_indices], graph_class_ids[test_indices])) == 0
print(f"{len(train_indices)} training graphs, {len(test_indices)} test graphs; no test graph is isomorphic to a training graph")

angle_regressor = RandomForestRegressor(n_estimators=RANDOM_FOREST_TREES, random_state=RANDOM_SEED)
angle_regressor.fit(graph_features[train_indices], best_angles_found[train_indices])
predicted_angles = angle_regressor.predict(graph_features[test_indices])
assert predicted_angles.shape == (len(test_indices), 2 * QAOA_LAYERS)

angle_error = np.array([symmetry_aware_angle_error(predicted_angles[position], best_angles_found[graph_index])
                        for position, graph_index in enumerate(test_indices)])
print("mean symmetry-aware absolute angle error on test graphs, gamma then beta per layer:", np.round(angle_error.mean(axis=0), 3))
print("for scale, a random guess would be off by about", np.round([GAMMA_PERIOD / 4] * QAOA_LAYERS + [BETA_PERIOD / 4] * QAOA_LAYERS, 3))
print("feature importances:")
for feature_name, importance in zip(FEATURE_NAMES, angle_regressor.feature_importances_):
    print(f"  {feature_name:24s} {importance:.3f}")

In [ ]:
# Evaluation on test graphs. For each one: a single run from the warm start, and EVALUATION_RANDOM_STARTS independent
# single runs from random starts. Each random run is one random start, not a best-of strategy.

warm_runs = []                       # one dict per test graph
random_runs = []                     # one list of dicts per test graph
warm_ratio_at_start = np.zeros(len(test_indices))
random_ratio_at_start = np.zeros((len(test_indices), EVALUATION_RANDOM_STARTS))

start_time = time.time()
for test_position, graph_index in enumerate(test_indices):
    edge_list = graph_edge_lists[graph_index]
    cut_values = cut_value_of_every_bitstring(edge_list, NODE_COUNT)
    max_cut = brute_force_max_cut[graph_index]
    target_cut = TARGET_FRACTION_OF_BEST * best_expected_cut_found[graph_index]

    warm_start = predicted_angles[test_position]
    warm_ratio_at_start[test_position] = expected_cut(warm_start, edge_list, cut_values, NODE_COUNT) / max_cut
    warm_runs.append(optimize_angles(warm_start, edge_list, cut_values, NODE_COUNT, target_cut))

    runs_for_this_graph = []
    for start_index in range(EVALUATION_RANDOM_STARTS):
        random_start = np.concatenate([
            random_generator.uniform(0, GAMMA_PERIOD, size=QAOA_LAYERS),
            random_generator.uniform(0, BETA_PERIOD, size=QAOA_LAYERS),
        ])
        random_ratio_at_start[test_position, start_index] = expected_cut(random_start, edge_list, cut_values, NODE_COUNT) / max_cut
        runs_for_this_graph.append(optimize_angles(random_start, edge_list, cut_values, NODE_COUNT, target_cut))
    random_runs.append(runs_for_this_graph)

    if (test_position + 1) % 25 == 0:
        print(f"evaluated {test_position + 1}/{len(test_indices)}  elapsed {time.time() - start_time:.0f}s")

# Unpack into arrays. Evaluations-to-target is NaN for runs that never reached it.
warm_evaluations = np.array([run["evaluations"] for run in warm_runs])
random_evaluations = np.array([[run["evaluations"] for run in runs] for runs in random_runs])
warm_to_target = np.array([np.nan if run["evaluations_to_target"] is None else run["evaluations_to_target"] for run in warm_runs])
random_to_target = np.array([[np.nan if run["evaluations_to_target"] is None else run["evaluations_to_target"] for run in runs] for runs in random_runs])
warm_ratio_at_end = np.array([run["best_cut"] for run in warm_runs]) / brute_force_max_cut[test_indices]
random_ratio_at_end = np.array([[run["best_cut"] for run in runs] for runs in random_runs]) / brute_force_max_cut[test_indices][:, None]
warm_at_cap = sum(run["hit_evaluation_cap"] for run in warm_runs)
random_at_cap = sum(run["hit_evaluation_cap"] for runs in random_runs for run in runs)

print()
print(f"RESULTS ON {len(test_indices)} TEST GRAPHS (unseen topologies), target = {TARGET_FRACTION_OF_BEST} of each graph's label cut")
print(f"evaluations until termination, median:        random {np.median(random_evaluations):.0f}   warm {np.median(warm_evaluations):.0f}")
print(f"runs that stopped at the evaluation cap:      random {random_at_cap} of {random_evaluations.size}   warm {warm_at_cap} of {warm_evaluations.size}")
print(f"runs that never reached the target:           random {np.isnan(random_to_target).sum()} of {random_to_target.size}   warm {np.isnan(warm_to_target).sum()} of {warm_to_target.size}")
print(f"evaluations to reach the target, median over runs that reached it:   random {np.nanmedian(random_to_target):.0f}   warm {np.nanmedian(warm_to_target):.0f}")
print(f"approximation ratio at the starting angles, mean:   random {random_ratio_at_start.mean():.3f}   warm {warm_ratio_at_start.mean():.3f}")
print(f"approximation ratio at termination, mean:           random {random_ratio_at_end.mean():.3f}   warm {warm_ratio_at_end.mean():.3f}")
print("termination messages seen:", sorted(set(run["message"] for run in warm_runs) | set(run["message"] for runs in random_runs for run in runs)))

In [ ]:
# Save everything a follow-up notebook needs, including the split and the evaluation arrays.

config = {
    "RANDOM_SEED": RANDOM_SEED,
    "NUMBER_OF_GRAPHS": NUMBER_OF_GRAPHS,
    "NODE_COUNT": NODE_COUNT,
    "EDGE_PROBABILITY_LOW": EDGE_PROBABILITY_LOW,
    "EDGE_PROBABILITY_HIGH": EDGE_PROBABILITY_HIGH,
    "QAOA_LAYERS": QAOA_LAYERS,
    "OPTIMIZER_MAX_EVALUATIONS": OPTIMIZER_MAX_EVALUATIONS,
    "OPTIMIZER_TOLERANCE": OPTIMIZER_TOLERANCE,
    "RANDOM_RESTARTS_PER_GRAPH": RANDOM_RESTARTS_PER_GRAPH,
    "GAMMA_PERIOD": GAMMA_PERIOD,
    "BETA_PERIOD": BETA_PERIOD,
    "TEST_FRACTION": TEST_FRACTION,
    "RANDOM_FOREST_TREES": RANDOM_FOREST_TREES,
    "EVALUATION_RANDOM_STARTS": EVALUATION_RANDOM_STARTS,
    "TARGET_FRACTION_OF_BEST": TARGET_FRACTION_OF_BEST,
    "FEATURE_NAMES": FEATURE_NAMES,
    "package_versions": package_versions,
}
dataset = {
    "config": config,
    "graph_edge_lists": graph_edge_lists,
    "graph_features": graph_features,
    "graph_class_ids": graph_class_ids,
    "best_angles_found": best_angles_found,
    "best_expected_cut_found": best_expected_cut_found,
    "brute_force_max_cut": brute_force_max_cut,
    "label_evaluations": label_evaluations,
    "train_indices": train_indices,
    "test_indices": test_indices,
    "predicted_angles": predicted_angles,
    "warm_runs": warm_runs,
    "random_runs": random_runs,
    "warm_ratio_at_start": warm_ratio_at_start,
    "random_ratio_at_start": random_ratio_at_start,
}
with open(OUTPUT_PATH, "wb") as output_file:
    pickle.dump(dataset, output_file)
print("saved", OUTPUT_PATH)

In [ ]:
# Poster figure: three panels. Evaluations until termination, evaluations to reach the target, and quality at the start.

figure, (left_axis, middle_axis, right_axis) = plt.subplots(1, 3, figsize=(15, 4))

left_axis.boxplot([random_evaluations.flatten(), warm_evaluations])
left_axis.set_xticks([1, 2])
left_axis.set_xticklabels(["random start", "warm start"])
left_axis.set_ylabel("objective evaluations")
left_axis.set_title("Evaluations until termination\n(runs end at different qualities)")

middle_axis.boxplot([random_to_target[~np.isnan(random_to_target)], warm_to_target[~np.isnan(warm_to_target)]])
middle_axis.set_xticks([1, 2])
middle_axis.set_xticklabels([f"random start\n({np.isnan(random_to_target).sum()} never reached)",
                             f"warm start\n({np.isnan(warm_to_target).sum()} never reached)"])
middle_axis.set_ylabel("objective evaluations")
middle_axis.set_title(f"Evaluations to reach {TARGET_FRACTION_OF_BEST} of the label cut")

right_axis.boxplot([random_ratio_at_start.flatten(), warm_ratio_at_start])
right_axis.set_xticks([1, 2])
right_axis.set_xticklabels(["random start", "warm start"])
right_axis.set_ylabel("expected cut / max cut")
right_axis.set_title("Quality before any optimization")

figure.suptitle(f"QAOA p={QAOA_LAYERS} warm start on {NODE_COUNT}-node random graphs, {len(test_indices)} unseen topologies")
figure.tight_layout()
figure.savefig(FIGURE_PATH, dpi=200)
plt.show()
print("saved", FIGURE_PATH)

## Shred-outs

The base run is a small simulator experiment on 8-node graphs with one compiler-free circuit. It says nothing about hardware or about quantum advantage, and it does not promise the warm start wins. Pick one direction below, or invent your own. Each is written as the question your poster answers and the number that answers it. If your extension sweeps a setting, choose the setting on a validation split carved out of the training graphs, then report the test number once.

1. **Better features, better model.** Does a richer feature set (full Laplacian spectrum, degree histogram, number of odd cycles) or a different model (gradient boosting, a small MLP) predict the angles more accurately, and does that translate into fewer evaluations to reach the target? Metric: median evaluations to target and the never-reached fraction on the test graphs.

2. **Transfer across graph sizes.** Train on 8-node graphs and warm-start QAOA on 10-, 12-, and 14-node graphs. Every helper in the circuit cell takes `node_count` as an argument, so build the new graphs, call `cut_value_of_every_bitstring(edges, node_count)` and `optimize_angles(..., node_count, target)` directly, and keep the regressor trained on the 8-node features. Does the warm start still help? Metric: evaluations to target as a function of node count. Watch the statevector size double with each added node.

3. **Deeper circuits.** Set `QAOA_LAYERS` to 2 or 3. The angle landscape gets harder and the labels get noisier. Does the warm start matter more or less at larger p? Metric: evaluations to target versus p. You may need more random restarts to trust the labels, and you should report how many labeling runs stopped at the evaluation cap.

4. **Angle symmetries and label quality.** The labels are wrapped using the three symmetries every graph has. Graphs whose vertex degrees all share a parity have one more: shifting gamma_k by pi, together with flipping the sign of every beta after layer k. COBYLA can also stop in a worse local optimum. Plot the labels against the degree-parity class of the graph. Are they clustered, and do the stragglers share a structure? Canonicalize the extra symmetry, raise the restart count, and see how much the angle error drops. Metric: symmetry-aware angle error before and after, and the fraction of labels that moved.

5. **Graph neural network instead of hand features.** Replace the feature table with a GNN that reads the graph directly and outputs the angles. Does it beat the random forest? Metric: angle error and evaluations to target, same test split.

6. **Predict the budget, not the angles.** Instead of angles, predict how many evaluations a graph will need, or which graphs QAOA will do poorly on. Metric: correlation between predicted and actual evaluations, or classification accuracy for "label approximation ratio below 0.8".

7. **Shots instead of exact statevectors.** Replace `Statevector(circuit).probabilities()` with a sampled estimate from `qiskit_aer` at a fixed shot count. Every objective evaluation is now noisy, and the target check needs an exact re-evaluation on the side. Does the warm start save more when every evaluation is expensive? Metric: evaluations to target at 100, 1000, and 10000 shots.

Whatever you choose, load `qaoa_warm_start_dataset.pkl` rather than regenerating it. The `config` dict inside tells you exactly which run produced it, and `train_indices` and `test_indices` are the split to keep.

In [ ]:
# Your extension starts here.
